<a href="https://colab.research.google.com/github/bezawitlegesse/Student-grade-analyzer/blob/main/week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 Lab: Small Data-Processing App
Pipeline: **load raw data → clean with functions → process with NumPy → report**

## 1. Imports and raw dataset
The raw data is messy on purpose: blanks, duplicates, bad numbers, out-of-range scores.

In [ ]:
import csv
import math
import numpy as np

RAW_CSV = "raw_scores.csv"

raw_rows = """name,score
 Alice ,85.5
Alice,85.5
Bob,0
Carla,92
,90
Dan,abc
Dana,150
Eve,nan
Frank,
Grace, 78
Hugo,64
Ivy,71.5
Jack,88
Kim,55
Liam,97
"""

# Write the raw dataset to a CSV so the notebook runs anywhere (Colab or VS Code)
with open(RAW_CSV, "w", newline="") as f:
    f.write(raw_rows)
print("Created", RAW_CSV)

Created raw_scores.csv


## 2. Load function

In [ ]:
def load_csv(path):
    """Read a CSV file and return a list of dictionaries (one per row)."""
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

raw_data = load_csv(RAW_CSV)
print(f"Loaded {len(raw_data)} raw rows")
raw_data[:3]

Loaded 15 raw rows


[{'name': ' Alice ', 'score': '85.5'},
 {'name': 'Alice', 'score': '85.5'},
 {'name': 'Bob', 'score': '0'}]

## 3. Cleaning functions

In [ ]:
def parse_score(value):
    """Return a valid score (0-100) as a float, or None if invalid."""
    text = "" if value is None else str(value).strip()
    if not text:
        return None
    try:
        score = float(text)
    except ValueError:
        return None
    if not math.isfinite(score) or score < 0 or score > 100:
        return None
    return score


def clean_data(data):
    """Strip names, drop invalid rows, remove duplicates."""
    cleaned = []
    seen = set()
    for row in data:
        raw_name = row.get("name")
        name = "" if raw_name is None else str(raw_name).strip()
        score = parse_score(row.get("score"))

        if not name or score is None:
            continue

        record = (name, score)
        if record not in seen:
            seen.add(record)
            cleaned.append({"name": name, "score": score})
    return cleaned


clean = clean_data(raw_data)
print(f"Kept {len(clean)} of {len(raw_data)} rows")
for row in clean:
    print(row)

Kept 9 of 15 rows
{'name': 'Alice', 'score': 85.5}
{'name': 'Bob', 'score': 0.0}
{'name': 'Carla', 'score': 92.0}
{'name': 'Grace', 'score': 78.0}
{'name': 'Hugo', 'score': 64.0}
{'name': 'Ivy', 'score': 71.5}
{'name': 'Jack', 'score': 88.0}
{'name': 'Kim', 'score': 55.0}
{'name': 'Liam', 'score': 97.0}


## 4. Process with NumPy

In [ ]:
def analyze(clean_rows):
    """Compute statistics, z-scores and letter grades with NumPy."""
    names = np.array([r["name"] for r in clean_rows])
    scores = np.array([r["score"] for r in clean_rows])

    stats = {
        "count": scores.size,
        "mean": scores.mean(),
        "median": np.median(scores),
        "std": scores.std(),
        "min": scores.min(),
        "max": scores.max(),
        "p25": np.percentile(scores, 25),
        "p75": np.percentile(scores, 75),
    }

    z_scores = (scores - stats["mean"]) / stats["std"]

    grades = np.select(
        [scores >= 90, scores >= 80, scores >= 70, scores >= 60],
        ["A", "B", "C", "D"],
        default="F",
    )
    return names, scores, z_scores, grades, stats


names, scores, z_scores, grades, stats = analyze(clean)

## 5. Report

In [ ]:
print("=== Summary statistics ===")
for key, value in stats.items():
    print(f"{key:>6}: {value:.2f}")

print("\n=== Results (best first) ===")
order = np.argsort(scores)[::-1]
print(f"{'Name':<8}{'Score':>7}{'Z':>7}  Grade")
for i in order:
    print(f"{names[i]:<8}{scores[i]:>7.1f}{z_scores[i]:>7.2f}  {grades[i]}")

print("\nTop 3:", ", ".join(names[order[:3]]))
print("Above average:", ", ".join(names[scores > stats["mean"]]))

=== Summary statistics ===
 count: 9.00
  mean: 70.11
median: 78.00
   std: 27.90
   min: 0.00
   max: 97.00
   p25: 64.00
   p75: 88.00

=== Results (best first) ===
Name      Score      Z  Grade
Liam       97.0   0.96  A
Carla      92.0   0.78  A
Jack       88.0   0.64  B
Alice      85.5   0.55  B
Grace      78.0   0.28  C
Ivy        71.5   0.05  C
Hugo       64.0  -0.22  D
Kim        55.0  -0.54  F
Bob         0.0  -2.51  F

Top 3: Liam, Carla, Jack
Above average: Alice, Carla, Grace, Ivy, Jack, Liam
